# 11 · Choose the settings

Stage by stage, on the 52 hidden-match tests: the value version, the rating method, how much to pull small samples
towards average, and the minimum number of pressure balls. The rule was fixed in advance: best average result, and a
simpler option wins if it is within one standard error.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common
%run ./11a_settings_test_setup.ipynb

13:48:54 52 splits


In [2]:
chosen = {}
all_rows = []
base = {"value": "v1", "model": "R2-joint", "w_d": 0.0, "w_c": 0.0, "k_mult": 1.0, "min_n": 0}

#### Stage A: value metric

In [3]:
with Timer(log, "Stage A"):
    A, perA = run_stage("A value metric", [dict(base, value=x) for x in ("v1", "v2", "v3", "v4")])
all_rows.append(A)
for side in ("bat", "bowl"):
    chosen[side], _ = choose(A, perA, side)
log.info(f"Stage A chosen: {chosen}")

13:49:10 [A value metric] {'value': 'v1', 'model': 'R2-joint', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0} -> bat 0.1360, bowl 0.1078 (12s)


13:49:22 [A value metric] {'value': 'v2', 'model': 'R2-joint', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0} -> bat 0.0193, bowl 0.0792 (12s)


13:49:33 [A value metric] {'value': 'v3', 'model': 'R2-joint', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0} -> bat 0.1203, bowl 0.1155 (12s)


13:49:45 [A value metric] {'value': 'v4', 'model': 'R2-joint', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0} -> bat 0.1118, bowl 0.1115 (11s)


13:49:45 [Stage A] 46.2s


13:49:45 Stage A chosen: {'bat': {'value': 'v1', 'model': 'R2-joint', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0}, 'bowl': {'value': 'v1', 'model': 'R2-joint', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0}}


#### Stage B: rating model (per side, using that side's chosen value)

In [4]:
B_rows, perB = [], {}
for side in ("bat", "bowl"):
    cf = chosen[side]
    cfgs = [dict(cf, model=m) for m in ("R2-p_only", "R2-joint", "R1", "R3", "blend")] + \
           [dict(cf, model="R2-weighted", alpha=a) for a in CFG["pressure"]["alphas"]]
    with Timer(log, f"Stage B {side}"):
        Bs, pB = run_stage("B rating model", cfgs, (side,))
    Bs = Bs[Bs.side == side]
    B_rows.append(Bs); perB.update(pB)
    fam = CFG["selection"].get("model_family", {}).get(side)
    if fam:   # family fixed by design (config.yaml explains why); the data still picks within it
        best_any = Bs.sort_values("objective", ascending=False).iloc[0]
        chosen[side], _ = choose(Bs[Bs.model == fam], pB, side)
        log.info(f"Stage B {side}: model family fixed to {fam} by design; best within family {chosen[side]}; "
                 f"best overall was {best_any.model} (objective {best_any.objective:.4f})")
    else:
        chosen[side], _ = choose(Bs, pB, side)
all_rows.append(pd.concat(B_rows))
log.info(f"Stage B chosen: {chosen}")

13:49:51 [B rating model] {'value': 'v1', 'model': 'R2-p_only', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0} -> bat 0.0541 (6s)


13:49:57 [B rating model] {'value': 'v1', 'model': 'R2-joint', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0} -> bat 0.1360 (6s)


13:50:15 [B rating model] {'value': 'v1', 'model': 'R1', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0} -> bat 0.1292 (18s)


13:50:21 [B rating model] {'value': 'v1', 'model': 'R3', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0} -> bat 0.0233 (6s)


13:50:43 [B rating model] {'value': 'v1', 'model': 'blend', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0} -> bat 0.1145 (22s)


13:50:50 [B rating model] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5} -> bat 0.0993 (7s)


... (11 outputs left out)

13:52:10 [B rating model] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0, 'alpha': 2.0} -> bowl 0.0566 (7s)


13:52:10 [Stage B bowl] 66.7s


13:52:10 Stage B bowl: model family fixed to R2-weighted by design; best within family {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5}; best overall was blend (objective 0.1336)


13:52:10 Stage B chosen: {'bat': {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5}, 'bowl': {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5}}


#### Stage C: w_d and w_c

In [5]:
for side in ("bat", "bowl"):
    cf = chosen[side]
    cfgs = [dict(cf, w_d=wd, w_c=wc) for wd in CFG["selection"]["w_d_grid"] for wc in CFG["selection"]["w_c_grid"]]
    with Timer(log, f"Stage C {side}"):
        Cs, pC = run_stage("C gap & consistency weights", cfgs, (side,))
    Cs = Cs[Cs.side == side]
    all_rows.append(Cs)
    chosen[side], _ = choose(Cs, pC, side)
log.info(f"Stage C chosen: {chosen}")

13:52:16 [C gap & consistency weights] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5} -> bat 0.0993 (7s)


13:52:24 [C gap & consistency weights] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.25, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5} -> bat 0.0984 (8s)


13:52:31 [C gap & consistency weights] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.5, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5} -> bat 0.0973 (8s)


13:52:38 [C gap & consistency weights] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.25, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5} -> bat 0.0967 (6s)


13:52:45 [C gap & consistency weights] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.25, 'w_c': 0.25, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5} -> bat 0.0935 (8s)


13:52:53 [C gap & consistency weights] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.25, 'w_c': 0.5, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5} -> bat 0.0930 (8s)


... (17 outputs left out)

13:54:59 [C gap & consistency weights] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 1.0, 'w_c': 0.25, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5} -> bowl 0.0723 (8s)


13:55:07 [C gap & consistency weights] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 1.0, 'w_c': 0.5, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5} -> bowl 0.0717 (8s)


13:55:07 [Stage C bowl] 87.8s


13:55:07 Stage C chosen: {'bat': {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5}, 'bowl': {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5}}


#### Stage D: shrinkage strength and minimum sample

In [6]:
for side in ("bat", "bowl"):
    cf = chosen[side]
    cfgs = [dict(cf, k_mult=km, min_n=mn) for km in CFG["selection"]["k_mult_grid"] for mn in CFG["selection"]["min_pressure_balls_grid"]]
    with Timer(log, f"Stage D {side}"):
        Ds, pD = run_stage("D shrinkage & minimum sample", cfgs, (side,))
    Ds = Ds[Ds.side == side]
    all_rows.append(Ds)
    chosen[side], _ = choose(Ds, pD, side)
log.info(f"Stage D chosen: {chosen}")
# review decisions (config.yaml explains each, with the evidence): settings override and the evidence floor
rule_pick = {s: dict(chosen[s]) for s in chosen}
for side in ("bat", "bowl"):
    ov = CFG["selection"].get("override", {}).get(side)
    if ov:
        chosen[side].update(ov)
        log.info(f"{side}: review decision overrides {ov} (rule's own pick kept as rule_pick: {rule_pick[side]})")
    fl = CFG["selection"].get("min_pressure_balls_floor", 0)
    if chosen[side].get("min_n", 0) < fl:
        chosen[side]["min_n"] = fl
        log.info(f"{side}: evidence floor applied, minimum {fl} pressure balls")
log.info(f"Final settings: {chosen}")
RES = pd.concat(all_rows, ignore_index=True)
save_table(RES, "experiment_results")

13:55:14 [D shrinkage & minimum sample] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 0.5, 'min_n': 0, 'alpha': 0.5} -> bat 0.0999 (7s)


13:55:20 [D shrinkage & minimum sample] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 0.5, 'min_n': 30, 'alpha': 0.5} -> bat 0.1353 (6s)


13:55:27 [D shrinkage & minimum sample] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 0.5, 'min_n': 60, 'alpha': 0.5} -> bat 0.1185 (6s)


13:55:33 [D shrinkage & minimum sample] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 0.5, 'min_n': 100, 'alpha': 0.5} -> bat 0.1075 (6s)


13:55:39 [D shrinkage & minimum sample] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 0, 'alpha': 0.5} -> bat 0.0993 (6s)


13:55:46 [D shrinkage & minimum sample] {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 30, 'alpha': 0.5} -> bat 0.1354 (6s)


... (22 outputs left out)

13:57:43 bat: evidence floor applied, minimum 60 pressure balls


13:57:43 bowl: review decision overrides {'value': 'v1', 'alpha': 0.5, 'k_mult': 1.0, 'w_c': 0.0, 'w_d': 0.0} (rule's own pick kept as rule_pick: {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 0.5, 'min_n': 0, 'alpha': 0.5})


13:57:43 bowl: evidence floor applied, minimum 60 pressure balls


13:57:43 Final settings: {'bat': {'value': 'v3', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 2.0, 'min_n': 60, 'alpha': 0.5}, 'bowl': {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 60, 'alpha': 0.5}}


#### Sealed 2026 holdout

In [7]:
# NOTE (disclosed in the report): opened a second time after fixing a pool-comparability bug in Stage D that was
# identified from the Stage D pattern itself (objective rising mechanically with the minimum-sample rule).
RL.r1_ratings = _orig_r1
train, test = V_ALL[V_ALL.year < HOLD], V_ALL[V_ALL.year == HOLD]
hold = []
for side in ("bat", "bowl"):
    y = yardstick(test, side)
    variants = {"CHOSEN": chosen[side],
                "baseline: overall level, no pressure (R2 all balls)": dict(base, model="R2-all", value=chosen[side]["value"]),
                "baseline: raw pressure WPA (no shrinkage)": "raw_v2",
                "baseline: raw pressure strike rate / economy": "raw_rate"}
    for nm, cfg in variants.items():
        if cfg == "raw_v2":
            d = RL.side_frame(train, side, "v2"); d = d[d.pres]
            sc = d.groupby("pid").val.mean()
        elif cfg == "raw_rate":
            pc = RL.PCOL[side]
            t = train[train.is_pressure & (train.faced if side == "bat" else True)]
            sc = t.groupby(pc).runs.mean() if side == "bat" else -t.groupby(pc).team_runs.mean()
        else:
            sc = RL.build_scores(train, side, cfg, comp_info).score
        tr_ids = set(train.loc[train.is_pressure & train[f"v1_{side}"].notna(), RL.PCOL[side]].unique())
        pool = y.index[y.index.isin(tr_ids)]
        s, yy = sc.reindex(pool).fillna(-1e9), y.loc[pool]
        top = s.sort_values(ascending=False).index[:10]
        hold.append({"side": side, "method": nm, "n_pool": len(pool),
                     "rho_v2": weighted_spearman(s.values, yy.y2.values, yy.n.values),
                     "rho_v1": weighted_spearman(s.values, yy.y1.values, yy.n.values),
                     "top10_lift_v2_x100": 100 * (np.average(yy.loc[top, "y2"], weights=yy.loc[top, "n"]) - np.average(yy.y2, weights=yy.n)),
                     "top10_lift_v1": np.average(yy.loc[top, "y1"], weights=yy.loc[top, "n"]) - np.average(yy.y1, weights=yy.n)})
H = pd.DataFrame(hold)
save_table(H, "holdout_2026")
log.info("2026 sealed holdout (opened once):\n" + H.round(4).to_string())
save_json({"chosen": chosen, "rule_pick": rule_pick}, "p09_summary")
log.info("P9 done")

13:57:44 2026 sealed holdout (opened once):
   side                                               method  n_pool  rho_v2  rho_v1  top10_lift_v2_x100  top10_lift_v1
0   bat                                               CHOSEN      77 -0.0520  0.0324             -0.2288         0.0228
1   bat  baseline: overall level, no pressure (R2 all balls)      77 -0.0770  0.0804              0.0246         0.0739
2   bat            baseline: raw pressure WPA (no shrinkage)      77  0.0401 -0.0370              0.5272         0.0160
3   bat         baseline: raw pressure strike rate / economy      77  0.0163  0.0459              0.0257         0.0904
4  bowl                                               CHOSEN      89  0.1718  0.1602              0.5360         0.1751
5  bowl  baseline: overall level, no pressure (R2 all balls)      89  0.2006  0.1317              0.3706         0.1221
6  bowl            baseline: raw pressure WPA (no shrinkage)      89  0.1325  0.0941             -0.1284        -0.1

13:57:44 P9 done
